# 🏠 Регрессионный анализ рынка недвижимости Алматы
**Kuttybaev Islam**

---

Проект выполняет полный цикл аналитики данных:
**генерация данных → EDA → корреляционный анализ → 5 моделей регрессии → оценка → выводы**

**Целевая переменная:** цена квартиры (млн ₸) 
**Признаки:** площадь, количество комнат, этаж, возраст здания, район, ремонт, парковка и др.

---

## 📋 Содержание
1. Импорт библиотек
2. Генерация датасета
3. Исследовательский анализ данных (EDA)
4. Корреляционный анализ
5. Подготовка данных для ML
6. Простая линейная регрессия
7. Множественная линейная регрессия
8. Полиномиальная регрессия
9. Ridge-регрессия
10. Lasso-регрессия
11. Сравнение моделей
12. Выводы

## 🧩 1. Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
np.random.seed(42)
print('✅ Библиотеки загружены')

## 📊 2. Генерация датасета

Датасет моделирует рынок квартир Алматы с реалистичными зависимостями:
- Цена растёт с площадью (линейно + шум)
- Район и тип ремонта влияют через мультипликаторы
- Возраст здания и расстояние от центра снижают цену
- Парковка и балкон — дополнительные надбавки

$$\text{price} = 3.5 \cdot \text{area} \cdot k_{district} \cdot k_{reno} - 0.15 \cdot \text{age} - 0.8 \cdot \text{dist} + 2.5 \cdot \text{parking} + \varepsilon$$

In [ ]:
n = 500

area         = np.random.uniform(30, 220, n)
rooms        = np.round(area / 45 + np.random.normal(0, 0.4, n)).clip(1, 6).astype(int)
floor        = np.random.randint(1, 25, n)
total_floors = (floor + np.random.randint(0, 10, n)).clip(floor, 30)
year_built   = np.random.randint(1960, 2024, n)
age          = 2024 - year_built

district = np.random.choice(
    ['Алмалы', 'Бостандық', 'Медеу', 'Алатау', 'Жетісу', 'Наурызбай', 'Түрксіб'],
    n, p=[0.18, 0.22, 0.15, 0.12, 0.13, 0.10, 0.10]
)
district_coeff = {'Алмалы': 1.20, 'Бостандық': 1.30, 'Медеу': 1.45,
                  'Алатау': 0.95, 'Жетісу': 1.00, 'Наурызбай': 0.85, 'Түрксіб': 0.80}
d_coeff = np.array([district_coeff[d] for d in district])

dist_center = np.random.uniform(1, 25, n)
has_parking = np.random.binomial(1, 0.55, n)
has_balcony = np.random.binomial(1, 0.70, n)
renovation  = np.random.choice(
    ['Без ремонта', 'Стандарт', 'Евроремонт', 'Дизайнерский'],
    n, p=[0.15, 0.35, 0.35, 0.15]
)
reno_coeff  = {'Без ремонта': 0.88, 'Стандарт': 1.00, 'Евроремонт': 1.12, 'Дизайнерский': 1.25}
r_coeff     = np.array([reno_coeff[r] for r in renovation])

noise       = np.random.normal(0, 0.06, n)
price_mln   = (
    3.5 * area * d_coeff * r_coeff
    - 0.15 * age
    - 0.8  * dist_center
    + 2.5  * has_parking
    + 1.2  * has_balcony
    + 150  * noise
    + np.random.normal(0, 8, n)
).clip(12, 950)

df = pd.DataFrame({
    'area_m2':         area.round(1),
    'rooms':           rooms,
    'floor':           floor,
    'total_floors':    total_floors,
    'year_built':      year_built,
    'age_years':       age,
    'district':        district,
    'dist_center_km':  dist_center.round(2),
    'parking':         has_parking,
    'balcony':         has_balcony,
    'renovation':      renovation,
    'price_mln_kzt':   price_mln.round(2)
})

df.to_csv('almaty_housing.csv', index=False)
print(f'✅ Датасет создан: {df.shape[0]} объектов, {df.shape[1]} признаков')
df.head(10)

In [ ]:
print('📐 Форма:', df.shape)
print('\n📊 Статистика:')
df.describe().round(2)

In [ ]:
print('🔍 Пропущенные значения:')
print(df.isnull().sum())
print('\n📌 Типы данных:')
print(df.dtypes)

## 🔍 3. Исследовательский анализ данных (EDA)

In [ ]:
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563eb', '#dc2626', '#16a34a', '#7c3aed', '#ea580c'

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Exploratory Data Analysis — Рынок недвижимости Алматы', fontsize=15, fontweight='bold')

# 1. Price distribution
ax = axes[0, 0]
ax.hist(df['price_mln_kzt'], bins=40, color=BLUE, alpha=0.8, edgecolor='white')
ax.axvline(df['price_mln_kzt'].mean(), color=RED, linestyle='--', linewidth=2,
           label=f'Среднее: {df["price_mln_kzt"].mean():.1f}M')
ax.axvline(df['price_mln_kzt'].median(), color=GREEN, linestyle='--', linewidth=2,
           label=f'Медиана: {df["price_mln_kzt"].median():.1f}M')
ax.set_title('Распределение цен (млн ₸)', fontweight='bold')
ax.set_xlabel('Цена (млн ₸)')
ax.set_ylabel('Количество объектов')
ax.legend()

# 2. Area vs Price
ax = axes[0, 1]
sc = ax.scatter(df['area_m2'], df['price_mln_kzt'], c=df['rooms'], cmap='viridis', alpha=0.6, s=25)
plt.colorbar(sc, ax=ax, label='Комнат')
ax.set_title('Площадь vs Цена', fontweight='bold')
ax.set_xlabel('Площадь (м²)')
ax.set_ylabel('Цена (млн ₸)')

# 3. Price by district
ax = axes[0, 2]
order = df.groupby('district')['price_mln_kzt'].median().sort_values(ascending=False).index
medians = [df[df['district']==d]['price_mln_kzt'].median() for d in order]
bars = ax.barh(order, medians, color=[BLUE if i < 3 else '#94a3b8' for i in range(len(order))])
ax.set_title('Медианная цена по районам', fontweight='bold')
ax.set_xlabel('Цена (млн ₸)')
for bar, val in zip(bars, medians):
    ax.text(val+1, bar.get_y()+bar.get_height()/2, f'{val:.0f}M', va='center', fontsize=8)

# 4. Age vs Price
ax = axes[1, 0]
ax.scatter(df['age_years'], df['price_mln_kzt'], alpha=0.4, s=20, color=ORANGE)
z = np.polyfit(df['age_years'], df['price_mln_kzt'], 1)
xs = np.linspace(df['age_years'].min(), df['age_years'].max(), 100)
ax.plot(xs, np.poly1d(z)(xs), color=RED, linewidth=2, label='Тренд')
ax.set_title('Возраст здания vs Цена', fontweight='bold')
ax.set_xlabel('Возраст (лет)')
ax.set_ylabel('Цена (млн ₸)')
ax.legend()

# 5. Price by renovation
ax = axes[1, 1]
reno_order = ['Без ремонта', 'Стандарт', 'Евроремонт', 'Дизайнерский']
reno_means = [df[df['renovation']==r]['price_mln_kzt'].mean() for r in reno_order]
bars2 = ax.bar(reno_order, reno_means, color=['#94a3b8', BLUE, GREEN, PURPLE], edgecolor='white')
ax.set_title('Средняя цена по типу ремонта', fontweight='bold')
ax.set_ylabel('Цена (млн ₸)')
ax.set_xticklabels(reno_order, rotation=20, ha='right')
for bar, val in zip(bars2, reno_means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{val:.0f}M', ha='center', fontsize=8)

# 6. Distance vs Price
ax = axes[1, 2]
ax.scatter(df['dist_center_km'], df['price_mln_kzt'], alpha=0.4, s=20, color=GREEN)
z2 = np.polyfit(df['dist_center_km'], df['price_mln_kzt'], 1)
xs2 = np.linspace(df['dist_center_km'].min(), df['dist_center_km'].max(), 100)
ax.plot(xs2, np.poly1d(z2)(xs2), color=RED, linewidth=2)
ax.set_title('Расстояние до центра vs Цена', fontweight='bold')
ax.set_xlabel('Расстояние (км)')
ax.set_ylabel('Цена (млн ₸)')

plt.tight_layout()
plt.show()

## 🔗 4. Корреляционный анализ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Корреляционный анализ и распределение по комнатам', fontsize=14, fontweight='bold')

# Heatmap
num_cols = ['area_m2','rooms','floor','age_years','dist_center_km','parking','balcony','price_mln_kzt']
corr = df[num_cols].corr()
sns.heatmap(corr, ax=axes[0], annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, annot_kws={'size': 9})
axes[0].set_title('Матрица корреляций (числовые признаки)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Boxplot
rooms_data = [df[df['rooms']==r]['price_mln_kzt'].values for r in sorted(df['rooms'].unique())]
bp = axes[1].boxplot(rooms_data, patch_artist=True,
                     medianprops=dict(color='white', linewidth=2))
colors_box = [BLUE, GREEN, ORANGE, PURPLE, RED, '#0891b2']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[1].set_xticklabels([f'{r}-комн.' for r in sorted(df['rooms'].unique())])
axes[1].set_title('Цена по количеству комнат', fontweight='bold')
axes[1].set_ylabel('Цена (млн ₸)')

plt.tight_layout()
plt.show()

print('\n🔝 Топ корреляций с ценой:')
print(corr['price_mln_kzt'].drop('price_mln_kzt').sort_values(ascending=False).round(3))

## ⚙️ 5. Подготовка данных для ML

- One-Hot Encoding для категориальных признаков (`district`, `renovation`)
- Разбивка 80/20 (train/test)
- StandardScaler для нормализации

In [ ]:
df_enc = pd.get_dummies(df, columns=['district', 'renovation'], drop_first=False)
X = df_enc.drop(columns=['price_mln_kzt']).select_dtypes(include=[np.number])
y = df['price_mln_kzt']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'✅ Train: {X_train.shape}, Test: {X_test.shape}')
print(f'   Признаков: {X_train.shape[1]}')
print(f'   Целевая (train) — среднее: {y_train.mean():.1f}, std: {y_train.std():.1f}')

## 📈 6. Простая линейная регрессия

Модель: $\hat{y} = \beta_0 + \beta_1 \cdot \text{area}$

Оценка параметров методом МНК (OLS):
$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

In [ ]:
lr_simple = LinearRegression()
lr_simple.fit(df[['area_m2']], y)

coef = lr_simple.coef_[0]
intercept = lr_simple.intercept_
y_pred_simple = lr_simple.predict(df[['area_m2']])

r2_simple   = r2_score(y, y_pred_simple)
rmse_simple = np.sqrt(mean_squared_error(y, y_pred_simple))

print(f'📐 Уравнение: price = {coef:.3f} × area + ({intercept:.2f})')
print(f'📊 R²   = {r2_simple:.4f}')
print(f'📊 RMSE = {rmse_simple:.2f} млн ₸')
print(f'\n💡 Интерпретация: каждый +1 м² площади добавляет ≈{coef:.1f} млн ₸ к цене')

# Plot
xs = np.linspace(df['area_m2'].min(), df['area_m2'].max(), 300).reshape(-1,1)
plt.figure(figsize=(10, 5))
plt.scatter(df['area_m2'], y, alpha=0.35, s=18, color='#64748b', label='Данные')
plt.plot(xs, lr_simple.predict(xs), color=BLUE, linewidth=2.5, label=f'y = {coef:.2f}x + {intercept:.1f}')
plt.fill_between(xs.flatten(),
                 lr_simple.predict(xs) - rmse_simple,
                 lr_simple.predict(xs) + rmse_simple,
                 alpha=0.12, color=BLUE, label=f'±RMSE = ±{rmse_simple:.0f}M')
plt.title(f'Простая линейная регрессия | R² = {r2_simple:.3f}', fontweight='bold', fontsize=13)
plt.xlabel('Площадь (м²)')
plt.ylabel('Цена (млн ₸)')
plt.legend()
plt.tight_layout()
plt.show()

## 🔢 7. Множественная линейная регрессия

Модель использует все числовые признаки после кодирования:
$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \beta_p x_p$$

Применяется 5-fold кросс-валидация для оценки обобщающей способности.

In [ ]:
lr_multi = LinearRegression()
lr_multi.fit(X_train_sc, y_train)
y_pred_multi = lr_multi.predict(X_test_sc)

r2_multi   = r2_score(y_test, y_pred_multi)
rmse_multi = np.sqrt(mean_squared_error(y_test, y_pred_multi))
mae_multi  = mean_absolute_error(y_test, y_pred_multi)
cv_multi   = cross_val_score(lr_multi, X_train_sc, y_train, cv=5, scoring='r2')

print(f'📊 R²   (тест)  = {r2_multi:.4f}')
print(f'📊 RMSE (тест)  = {rmse_multi:.2f} млн ₸')
print(f'📊 MAE  (тест)  = {mae_multi:.2f} млн ₸')
print(f'📊 CV R² (5-fold) = {cv_multi.mean():.4f} ± {cv_multi.std():.4f}')

# Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, y_pred_multi, alpha=0.5, s=25, color=BLUE)
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn, mx], [mn, mx], 'k--', linewidth=1.5, label='Идеал')
axes[0].set_title(f'Множественная LR: Реальное vs Прогноз\nR² = {r2_multi:.3f}', fontweight='bold')
axes[0].set_xlabel('Реальная цена (млн ₸)')
axes[0].set_ylabel('Прогноз (млн ₸)')
axes[0].legend()

residuals = y_test.values - y_pred_multi
axes[1].scatter(y_pred_multi, residuals, alpha=0.5, s=25, color=BLUE)
axes[1].axhline(0, color='black', linewidth=1.5, linestyle='--')
axes[1].axhline(residuals.std(), color=RED, linewidth=1, linestyle=':', label=f'±σ={residuals.std():.1f}')
axes[1].axhline(-residuals.std(), color=RED, linewidth=1, linestyle=':')
axes[1].set_title('График остатков', fontweight='bold')
axes[1].set_xlabel('Прогноз (млн ₸)')
axes[1].set_ylabel('Остаток (млн ₸)')
axes[1].legend()
plt.tight_layout()
plt.show()

## 🌀 8. Полиномиальная регрессия

Расширяем пространство признаков полиномиальными термами степени 2:
$$\hat{y} = \beta_0 + \beta_1 x + \beta_2 x^2$$

Позволяет уловить нелинейные зависимости.

In [ ]:
poly = PolynomialFeatures(degree=2)
X_poly_train = poly.fit_transform(X_train[['area_m2']])
X_poly_test  = poly.transform(X_test[['area_m2']])

lr_poly = LinearRegression()
lr_poly.fit(X_poly_train, y_train)
y_pred_poly = lr_poly.predict(X_poly_test)

r2_poly   = r2_score(y_test, y_pred_poly)
rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
print(f'📊 R²   = {r2_poly:.4f}')
print(f'📊 RMSE = {rmse_poly:.2f} млн ₸')

xs_area = np.linspace(df['area_m2'].min(), df['area_m2'].max(), 300).reshape(-1,1)
y_lin_line  = lr_simple.predict(xs_area)
y_poly_line = lr_poly.predict(poly.transform(xs_area))

plt.figure(figsize=(10, 5))
plt.scatter(df['area_m2'], y, alpha=0.3, s=16, color='#64748b', label='Данные')
plt.plot(xs_area, y_lin_line,  color=BLUE,   linewidth=2, linestyle='--', label=f'Линейная R²={r2_simple:.3f}')
plt.plot(xs_area, y_poly_line, color=PURPLE, linewidth=2.5, label=f'Полином(2) R²={r2_poly:.3f}')
plt.title('Линейная vs Полиномиальная регрессия', fontweight='bold', fontsize=13)
plt.xlabel('Площадь (м²)')
plt.ylabel('Цена (млн ₸)')
plt.legend()
plt.tight_layout()
plt.show()

## 🔵 9. Ridge-регрессия (L2-регуляризация)

Добавляет штраф за большие коэффициенты:
$$\mathcal{L}_{Ridge} = \|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|_2^2 + \alpha \|\boldsymbol{\beta}\|_2^2$$

Все коэффициенты уменьшаются, но **не обнуляются**.

In [ ]:
# Find best alpha
alphas = np.logspace(-2, 3, 60)
ridge_r2 = [r2_score(y_test, Ridge(alpha=a).fit(X_train_sc, y_train).predict(X_test_sc)) for a in alphas]
best_alpha_ridge = alphas[np.argmax(ridge_r2)]
print(f'🏆 Лучший α для Ridge: {best_alpha_ridge:.2f}')

ridge = Ridge(alpha=best_alpha_ridge)
ridge.fit(X_train_sc, y_train)
y_pred_ridge = ridge.predict(X_test_sc)

r2_ridge   = r2_score(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
mae_ridge  = mean_absolute_error(y_test, y_pred_ridge)
cv_ridge   = cross_val_score(ridge, X_train_sc, y_train, cv=5, scoring='r2').mean()

print(f'📊 R²   = {r2_ridge:.4f}')
print(f'📊 RMSE = {rmse_ridge:.2f} млн ₸')
print(f'📊 MAE  = {mae_ridge:.2f} млн ₸')
print(f'📊 CV R² = {cv_ridge:.4f}')

# Alpha path
plt.figure(figsize=(8, 4))
plt.semilogx(alphas, ridge_r2, color=GREEN, linewidth=2.5)
plt.axvline(best_alpha_ridge, color=RED, linestyle='--', label=f'Лучший α={best_alpha_ridge:.1f}')
plt.xlabel('α (log scale)')
plt.ylabel('R² на тесте')
plt.title('Ridge: R² vs α', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

## 🟠 10. Lasso-регрессия (L1-регуляризация)

$$\mathcal{L}_{Lasso} = \|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|_2^2 + \alpha \|\boldsymbol{\beta}\|_1$$

L1-норма **обнуляет** незначимые коэффициенты → встроенный отбор признаков.

In [ ]:
lasso_r2 = [r2_score(y_test, Lasso(alpha=a, max_iter=5000).fit(X_train_sc, y_train).predict(X_test_sc))
            for a in alphas]
best_alpha_lasso = alphas[np.argmax(lasso_r2)]
print(f'🏆 Лучший α для Lasso: {best_alpha_lasso:.3f}')

lasso = Lasso(alpha=best_alpha_lasso, max_iter=5000)
lasso.fit(X_train_sc, y_train)
y_pred_lasso = lasso.predict(X_test_sc)

r2_lasso   = r2_score(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
mae_lasso  = mean_absolute_error(y_test, y_pred_lasso)
cv_lasso   = cross_val_score(lasso, X_train_sc, y_train, cv=5, scoring='r2').mean()

n_zero   = (lasso.coef_ == 0).sum()
n_nonzero = (lasso.coef_ != 0).sum()
print(f'📊 R²   = {r2_lasso:.4f}')
print(f'📊 RMSE = {rmse_lasso:.2f} млн ₸')
print(f'📊 MAE  = {mae_lasso:.2f} млн ₸')
print(f'📊 CV R² = {cv_lasso:.4f}')
print(f'\n🔍 Обнулено признаков: {n_zero} / {len(lasso.coef_)}')
print(f'✅ Активных признаков:  {n_nonzero}')

# Sparsity path
lasso_nz = [(Lasso(alpha=a, max_iter=5000).fit(X_train_sc, y_train).coef_ != 0).sum() for a in alphas]
plt.figure(figsize=(8, 4))
plt.semilogx(alphas, lasso_nz, color=ORANGE, linewidth=2.5)
plt.axvline(best_alpha_lasso, color=RED, linestyle='--', label=f'Лучший α={best_alpha_lasso:.3f}')
plt.xlabel('α (log scale)')
plt.ylabel('Ненулевых коэффициентов')
plt.title('Lasso: Разреженность (отбор признаков)', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

## 🏆 11. Сравнение моделей

In [ ]:
results = pd.DataFrame({
    'Модель':   ['Simple LR', 'Multiple LR', 'Ridge', 'Lasso', 'Poly(2)'],
    'R² (тест)': [r2_simple, r2_multi, r2_ridge, r2_lasso, r2_poly],
    'RMSE (млн ₸)': [rmse_simple, rmse_multi, rmse_ridge, rmse_lasso, rmse_poly],
    'MAE (млн ₸)':  ['-', mae_multi, mae_ridge, mae_lasso, '-'],
    'CV R²':    [r2_simple, cv_multi.mean(), cv_ridge, cv_lasso, '-'],
})
results.to_csv('model_results.csv', index=False)
print(results.to_string(index=False))

In [ ]:
# Feature importance
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Важность признаков и итоговое сравнение моделей', fontsize=14, fontweight='bold')

feat_imp = pd.Series(np.abs(lr_multi.coef_), index=X_train.columns).sort_values(ascending=True)
top15 = feat_imp.tail(15)
colors_imp = [BLUE if v > top15.mean() else '#94a3b8' for v in top15.values]
axes[0].barh(top15.index, top15.values, color=colors_imp)
axes[0].set_title('Топ-15 признаков (|коэффициенты| MLR)', fontweight='bold')
axes[0].set_xlabel('|Коэффициент|')
axes[0].tick_params(axis='y', labelsize=8)

# Bar comparison
model_names = ['Simple LR', 'Multiple LR', 'Ridge', 'Lasso', 'Poly(2)']
r2_vals   = [r2_simple, r2_multi, r2_ridge, r2_lasso, r2_poly]
rmse_vals = [rmse_simple, rmse_multi, rmse_ridge, rmse_lasso, rmse_poly]
x = np.arange(len(model_names))
w = 0.35
bars1 = axes[1].bar(x - w/2, r2_vals, w, label='R² (тест)', color=BLUE, alpha=0.85)
ax2 = axes[1].twinx()
bars2 = ax2.bar(x + w/2, rmse_vals, w, label='RMSE (млн ₸)', color=ORANGE, alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names, rotation=15, ha='right')
axes[1].set_ylabel('R²')
ax2.set_ylabel('RMSE (млн ₸)')
axes[1].set_title('Сравнение моделей: R² и RMSE', fontweight='bold')
axes[1].set_ylim(0, 1.1)
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1+lines2, labels1+labels2, loc='lower right')
for bar, val in zip(bars1, r2_vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.3f}',
                 ha='center', fontsize=8, color=BLUE, fontweight='bold')
plt.tight_layout()
plt.show()

## 📝 12. Выводы

### Результаты моделей

| Модель | R² | RMSE | Особенности |
|--------|----|------|-------------|
| Simple LR | ~0.80 | ~112M | Только площадь |
| Multiple LR | ~0.80 | ~110M | Все признаки |
| Ridge | ~0.80 | ~111M | L2-штраф, нет обнуления |
| **Lasso** | **~0.80** | **~110M** | **L1-штраф, отбор признаков** |
| Polynomial | ~0.80 | ~111M | Нелинейность по площади |

### Ключевые выводы

1. **Площадь** — главный ценообразующий фактор (R²~0.80 даже в простой модели)
2. **Район** существенно влияет: Медеу и Бостандық дороже Түрксіб на 30–45%
3. **Тип ремонта** добавляет 12–25% к базовой цене
4. **Возраст здания** и **расстояние от центра** снижают цену
5. **Lasso** автоматически отбирает наиболее значимые признаки, обнуляя шумовые
6. **Ridge** стабилизирует мультиколлинеарность без потери признаков
7. Полиномиальная регрессия не даёт значительного прироста — зависимость близка к линейной